# FLUXION — UI Mockup Generator
> FLUX.1-schnell da Kaggle Model Hub (zero token HF) | P100 16GB
> Output: 8 PNG in /kaggle/working/

In [ ]:
# CELLA 1 — INSTALL (versioni pinned per compatibilità bitsandbytes + diffusers)
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'diffusers>=0.30.3',
     'transformers==4.46.3',   # pin: 5.x incompatibile con bitsandbytes (Int8Params error)
     'accelerate>=0.34.0',
     'sentencepiece',
     'protobuf',
     'safetensors',
     'bitsandbytes>=0.43.0',
    ],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('pip install fallito')
print('Dipendenze installate OK')

In [ ]:
# CELLA 2 — VERIFICA GPU + PATH MODELLO
import torch, os
from pathlib import Path
import pkg_resources

for pkg in ['diffusers', 'transformers', 'accelerate']:
    print(f'  {pkg}: {pkg_resources.get_distribution(pkg).version}')
print(f'torch: {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU non disponibile')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Path confermato dal web UI Kaggle
MODEL_PATH = '/kaggle/input/models/sanderland/flux.1-schnell/pytorch/hf/1'
p = Path(MODEL_PATH)
if not p.exists():
    # Fallback: cerca il modello ovunque
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('schnell' in f.lower() for f in files + dirs):
            print(f'Trovato in: {root}')
            MODEL_PATH = root
            break
    else:
        raise RuntimeError(f'Modello non trovato. Contenuto /kaggle/input/: {list(Path("/kaggle/input").rglob("*"))[:20]}')

print(f'Modello: {MODEL_PATH}')
print(f'Contenuto: {os.listdir(MODEL_PATH)}')

In [ ]:
# CELLA 3 — PRE-ENCODING + CARICA PIPELINE
# v19: ZERO bitsandbytes → zero cublasLt errors su P100 CC6.0
# Strategia:
#   1. T5+CLIP su CPU → encode 8 prompt → salva embeddings → libera T5+CLIP (~10GB RAM)
#   2. Carica transformer float16 + enable_sequential_cpu_offload (1 blocco in VRAM at a time)
#   3. Genera con prompt_embeds pre-calcolati (no text encoder durante inference)
import gc, torch, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from diffusers import FluxPipeline
from transformers import T5EncoderModel, CLIPTextModel, CLIPTokenizer, AutoTokenizer

# ─── PROMPTS (definiti qui per pre-encoding) ────────────────────────────────
W, H, STEPS = 1024, 576, 4
PROMPTS = {
    '01-scheda-parrucchiere.png': (
        'FLUXION Italian hair salon software desktop app, dark mode UI slate-900 background, '
        'client profile card showing haircut history timeline, color treatment records with '
        'hair color swatches palette, product allergy warnings red badge, appointment list, '
        'teal-500 accent color, glassmorphism cards subtle blur, Italian labels '
        'Nome Cliente Servizio Data Colore Allergie, professional 2026 UI screenshot', 11),
    '02-scheda-fitness.png': (
        'FLUXION Italian gym fitness management software desktop app, dark mode UI slate-900, '
        'client fitness profile with circular BMI gauge teal, body measurements progress bars, '
        'workout plan weekly schedule grid, weight history line chart, '
        'Italian labels Obiettivi Misurazioni Progressi Scheda Allenamento, '
        'teal accent professional sports desktop app 2026 UI screenshot', 22),
    '03-scheda-medica.png': (
        'FLUXION Italian medical clinic software desktop app, dark mode UI slate-900, '
        'patient medical record card, anamnesi medical history section, '
        'medications list with dosage, allergy warnings red badges, '
        'GDPR consent checkbox green indicator, appointment history timeline, '
        'Italian labels Patologie Farmaci Allergie Consenso GDPR Anamnesi, '
        'clinical professional desktop UI 2026 screenshot', 33),
    '04-scheda-estetica.png': (
        'FLUXION Italian beauty salon aesthetic software desktop app, dark mode slate-900, '
        'client beauty profile card, cosmetic treatment history list, '
        'skin type assessment cards icons, product preferences section, '
        'before-after photo placeholders side by side, loyalty points badge teal, '
        'Italian labels Trattamenti Prodotti Preferenze Punti Fedelta, '
        'luxury professional desktop app 2026 UI screenshot', 44),
    '05-scheda-veicoli.png': (
        'FLUXION Italian auto repair workshop software desktop app, dark mode slate-900, '
        'vehicle client card showing car targa plate modello anno, '
        'maintenance service history timeline vertical, '
        'next service reminder orange alert badge, diagnostic notes textarea, '
        'replaced parts list checkmarks, '
        'Italian labels Veicolo Targa Manutenzione Revisione Prossimo Tagliando, '
        'professional workshop management desktop 2026 UI screenshot', 55),
    '06-dashboard.png': (
        'FLUXION Italian PMI business management software dashboard, dark mode slate-900, '
        'KPI metric cards oggi settimana mese revenue in teal green, '
        'weekly appointment calendar right panel colored blocks, '
        'left sidebar navigation icons Clienti Calendario Marketing Operatori, '
        'recent bookings list table, quick action buttons teal, '
        'professional Italian desktop management app 2026 UI screenshot', 66),
    '07-calendario.png': (
        'FLUXION Italian booking calendar desktop app, dark mode slate-900, '
        'weekly calendar view with colored appointment cards per staff member, '
        'multiple operator columns left to right with color coding, '
        'time slots 8am to 8pm vertical axis, client name service on each card, '
        'teal new appointment button top right corner, '
        'Italian professional salon booking calendar 2026 UI screenshot', 77),
    '08-voice-sara.png': (
        'FLUXION Italian AI voice assistant Sara interface desktop app, dark mode slate-900, '
        'voice waveform animation teal glowing center, '
        'conversation transcript panel showing Italian booking dialogue, '
        'booking confirmation card with cliente orario servizio fields, '
        'microphone icon animated pulsing, '
        'Italian labels Prenotazione Confermata Sara Ascoltando, '
        'futuristic voice AI booking assistant 2026 UI screenshot', 88),
}

# ─── STEP 1: Pre-encode tutti i prompt con T5+CLIP su CPU ───────────────────
# Liberiamo entrambi PRIMA di caricare il transformer (~24GB)
# RAM totale T5+CLIP: ~10.5GB → liberata dopo encoding
print('Caricamento T5-XXL su CPU (float16, ~10GB RAM)...')
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_PATH, subfolder='tokenizer_2', local_files_only=True)
t5 = T5EncoderModel.from_pretrained(
    MODEL_PATH, subfolder='text_encoder_2',
    torch_dtype=torch.float16, low_cpu_mem_usage=True, local_files_only=True)

print('Caricamento CLIP su CPU (float16, ~235MB)...')
tokenizer = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder='tokenizer', local_files_only=True)
clip = CLIPTextModel.from_pretrained(
    MODEL_PATH, subfolder='text_encoder',
    torch_dtype=torch.float16, low_cpu_mem_usage=True, local_files_only=True)

print('Encoding prompts su CPU...')
all_embeds = {}
for name, (prompt, _) in PROMPTS.items():
    # T5: [1, 256, 4096]
    t5_tok = tokenizer_2(prompt, return_tensors='pt', padding='max_length',
                          max_length=256, truncation=True, add_special_tokens=True)
    with torch.no_grad():
        pe = t5(input_ids=t5_tok.input_ids).last_hidden_state.to(torch.float16).cpu()
    # CLIP: [1, 768]
    clip_tok = tokenizer(prompt, return_tensors='pt', padding='max_length',
                          max_length=77, truncation=True)
    with torch.no_grad():
        ppe = clip(input_ids=clip_tok.input_ids).pooler_output.to(torch.float16).cpu()
    all_embeds[name] = (pe, ppe)
    print(f'  OK: {name}')

del t5, clip, tokenizer, tokenizer_2
gc.collect()
print(f'T5+CLIP liberati | VRAM libera: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f}GB')

# ─── STEP 2: Carica pipeline SENZA text encoders (già encodati) ─────────────
# Transformer float16 + sequential_cpu_offload: 1 blocco (~400MB) in VRAM alla volta
# ZERO bitsandbytes → ZERO cublasLt errors su P100 CC6.0
print('Caricamento FluxPipeline (float16, no text encoders)...')
pipe = FluxPipeline.from_pretrained(
    MODEL_PATH,
    text_encoder=None,    # Skip — pre-encodato
    text_encoder_2=None,  # Skip — pre-encodato
    tokenizer=None,
    tokenizer_2=None,
    torch_dtype=torch.float16,
    local_files_only=True,
)
pipe.enable_sequential_cpu_offload()
print('Pipeline OK — sequential CPU offload attivo (peak VRAM: ~400MB per blocco)')

In [ ]:
# CELLA 4 — GENERA 8 MOCKUP UI FLUXION
# Usa prompt_embeds pre-calcolati (nessuna chiamata a T5/CLIP durante inference)
import torch, os
from pathlib import Path
from PIL import Image

OUT = Path('/kaggle/working/')
results = []

for filename, (prompt, seed) in PROMPTS.items():
    print(f'Generazione {filename}...')
    try:
        pe, ppe = all_embeds[filename]
        g = torch.Generator(device='cpu').manual_seed(seed)
        # pipe.encode_prompt gestisce il .to(device) automaticamente internamente
        result = pipe(
            prompt=None,
            prompt_embeds=pe,              # [1, 256, 4096] — T5 pre-calcolato
            pooled_prompt_embeds=ppe,      # [1, 768] — CLIP pre-calcolato
            width=W,
            height=H,
            num_inference_steps=STEPS,
            guidance_scale=0.0,
            generator=g,
            max_sequence_length=256,
        )
        img = result.images[0]
        path = OUT / filename
        img.save(str(path))
        size_kb = os.path.getsize(str(path)) // 1024
        print(f'  OK — {img.size[0]}x{img.size[1]} | {size_kb} KB')
        results.append(filename)
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  ERRORE: {e}')

print(f'\nCompletato: {len(results)}/{len(PROMPTS)} immagini')
for f in results:
    print(f'  {f}')

In [ ]:
# CELLA 5 — PREVIEW
from IPython.display import display, Image as IPImage
import os

for f in sorted(os.listdir('/kaggle/working/')):
    if f.endswith('.png'):
        print(f)
        display(IPImage(filename=f'/kaggle/working/{f}', width=700))